# SoftCAM-Transformer v4 — Run C3 (diagnostic : sweep gate_strength à l'inférence)

**Objectif** : isoler la cause des échecs C1/C2 en évaluant le checkpoint **v4_runC2_final.pt** avec différentes valeurs de `gate_strength` ∈ {0, 0.1, 0.25, 0.5, 0.75, 1.0} **sans réentraîner**.

| Hypothèse | Test |
|--|--|
| Le Transformer interne est sain, c'est le gate qui casse generate() | À `gate_strength=0` → gate=1 strict → R² doit être positif |
| Le gate déstabilise progressivement | R² monotone décroissant à mesure que s augmente |
| Existe un sweet spot intermédiaire | R² maximal à s∈(0, 1) — viable comme inférence finale |

**Coût** : ~10 min Colab, **zéro entraînement**.

**Référence checkpoint** : RunC2 (`R²=-5.36`, `Spearman=-0.63`, `gate_deviation=0.40` à s=1).

> Avant de lancer : Runtime → Change runtime type → T4 GPU (ou CPU).

## 1 — Setup

In [ ]:
import subprocess
gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(gpu.stdout if gpu.returncode == 0 else 'Pas de GPU — CPU suffira.')

In [ ]:
%%capture
!pip install -q transformers datasets "gluonts[torch]" accelerate evaluate scipy scikit-learn tqdm openpyxl ujson

## 2 — Récupération du code

In [ ]:
import os, sys

REPO_URL = 'https://github.com/theblackmamba08/recherche-m2-xai-faas.git'
REPO_DIR = '/content/recherche-m2-xai-faas'

ipy = get_ipython()
if not os.path.isdir(REPO_DIR):
    ipy.system(f'git clone {REPO_URL} {REPO_DIR}')
else:
    ipy.system(f'git -C {REPO_DIR} pull')

if f'{REPO_DIR}/code' not in sys.path:
    sys.path.insert(0, f'{REPO_DIR}/code')

print('Repo prêt.')

## 3 — Imports + config (identiques RunC2)

In [ ]:
import random, json, gc
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
from functools import lru_cache, partial
from pathlib import Path

SEED = 998
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch : {torch.__version__}')
print(f'Device  : {device}')

FREQ              = '1T'
PREDICTION_LENGTH = 120
CONTEXT_LENGTH    = 240
D_MODEL           = 32
N_HEADS           = 2
ENCODER_LAYERS    = 4
DECODER_LAYERS    = 4
EMBEDDING_DIM     = [2]
DROPOUT           = 0.1
BATCH_SIZE_TEST   = 64
USE_EVIDENCE_LAYER   = True
ALPHA_L1             = 0.0
BETA_L2              = 1e-3
GAMMA_ENTROPY_TARGET = 1e-3

FAYAM_R2, FAYAM_SPEAR = 0.3701, 0.9201
B5_R2,    B5_SPEAR    = 0.7563, 0.9870
C1_R2,    C1_SPEAR    = -5.2913, -0.5060
C2_R2,    C2_SPEAR    = -5.3643, -0.6258
GATE_R2,  GATE_SPEAR  = 0.30, 0.85

CLUSTER_ID = 4
RUN_NAME   = f'softcam-cluster{CLUSTER_ID}-v4-runC3-sweep'

GATE_STRENGTH_VALUES = [0.00, 0.10, 0.25, 0.50, 0.75, 1.00]

print(f'Run    : {RUN_NAME}')
print(f'Sweep s: {GATE_STRENGTH_VALUES}')
print(f'Cible  : R²(s=0) > 0 prouverait que le Transformer interne est sain')

## 4 — Google Drive + checkpoint RunC2

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CKPT_PATH = '/content/drive/MyDrive/m2-xai-faas/experiments/softcam-cluster4-v4-runC2/checkpoints/v4_runC2_final.pt'
DRIVE_BASE = f'/content/drive/MyDrive/m2-xai-faas/experiments/{RUN_NAME}'
for subdir in ['results', 'figures']:
    os.makedirs(f'{DRIVE_BASE}/{subdir}', exist_ok=True)

if not os.path.isfile(CKPT_PATH):
    raise FileNotFoundError(f'Checkpoint RunC2 introuvable : {CKPT_PATH}')

ckpt_size_mb = os.path.getsize(CKPT_PATH) / 1e6
print(f'Checkpoint RunC2 : {CKPT_PATH}')
print(f'Taille           : {ckpt_size_mb:.1f} MB')
print(f'Dossier sweep    : {DRIVE_BASE}')

## 5 — Chargement Cluster 4 (test set)

In [ ]:
from datasets import Dataset

DATA_DIR   = Path('/content/drive/MyDrive/Recherche/Datasets')
START_DATE = '2021-01-01 00:00:00'

df = pd.read_csv(DATA_DIR / f'cluster_{CLUSTER_ID}.csv', index_col='Function')
all_series = []
for func_id, row in df.iterrows():
    all_series.append({
        'function_id': int(func_id),
        'cluster':     CLUSTER_ID,
        'target_full': row.values.astype(np.float32),
    })

train_rows, test_rows = [], []
for ts_idx, s in enumerate(all_series):
    base = {'start': START_DATE, 'feat_static_cat': [ts_idx],
            'cluster': s['cluster'], 'function_id': s['function_id']}
    train_rows.append({**base, 'target': s['target_full'][:-PREDICTION_LENGTH].tolist()})
    test_rows.append( {**base, 'target': s['target_full'].tolist()})

train_dataset = Dataset.from_list(train_rows)
test_dataset  = Dataset.from_list(test_rows)
print(f'Cluster {CLUSTER_ID} — {len(all_series)} fonctions  (test={len(test_dataset)})')

## 6 — Pipeline GluonTS (backtest)

In [ ]:
from gluonts.time_feature import get_lags_for_frequency, time_features_from_frequency_str
from gluonts.dataset.field_names import FieldName
from gluonts.transform import (
    AddAgeFeature, AddObservedValuesIndicator, AddTimeFeatures,
    AsNumpyArray, Chain, InstanceSplitter,
    RemoveFields, ValidationSplitSampler,
    VstackFeatures, RenameFields,
)
from gluonts.dataset.loader import as_stacked_batches

lags_sequence = get_lags_for_frequency(FREQ)
time_features = time_features_from_frequency_str(FREQ)

@lru_cache(10_000)
def convert_to_pandas_period(date, freq):
    return pd.Period(date, freq)

def transform_start_field(batch, freq):
    batch['start'] = [convert_to_pandas_period(d, freq) for d in batch['start']]
    return batch

for ds in (train_dataset, test_dataset):
    ds.set_transform(partial(transform_start_field, freq=FREQ))

def create_transformation(freq, config):
    remove_field_names = []
    if config.num_static_real_features == 0:
        remove_field_names.append(FieldName.FEAT_STATIC_REAL)
    if config.num_dynamic_real_features == 0:
        remove_field_names.append(FieldName.FEAT_DYNAMIC_REAL)
    if config.num_static_categorical_features == 0:
        remove_field_names.append(FieldName.FEAT_STATIC_CAT)
    return Chain(
        [RemoveFields(field_names=remove_field_names)]
        + ([AsNumpyArray(field=FieldName.FEAT_STATIC_CAT, expected_ndim=1, dtype=int)]
           if config.num_static_categorical_features > 0 else [])
        + [AsNumpyArray(field=FieldName.TARGET, expected_ndim=1),
           AddObservedValuesIndicator(target_field=FieldName.TARGET,
                                      output_field=FieldName.OBSERVED_VALUES),
           AddTimeFeatures(start_field=FieldName.START, target_field=FieldName.TARGET,
                           output_field=FieldName.FEAT_TIME,
                           time_features=time_features_from_frequency_str(freq),
                           pred_length=config.prediction_length),
           AddAgeFeature(target_field=FieldName.TARGET, output_field=FieldName.FEAT_AGE,
                         pred_length=config.prediction_length, log_scale=True),
           VstackFeatures(output_field=FieldName.FEAT_TIME,
                          input_fields=[FieldName.FEAT_TIME, FieldName.FEAT_AGE]),
           RenameFields(mapping={
               FieldName.FEAT_STATIC_CAT: 'static_categorical_features',
               FieldName.FEAT_TIME:       'time_features',
               FieldName.TARGET:          'values',
               FieldName.OBSERVED_VALUES: 'observed_mask',
           })]
    )

def create_instance_splitter(config, mode):
    sampler = ValidationSplitSampler(min_future=config.prediction_length)
    return InstanceSplitter(
        target_field='values', is_pad_field=FieldName.IS_PAD,
        start_field=FieldName.START, forecast_start_field=FieldName.FORECAST_START,
        instance_sampler=sampler,
        past_length=config.context_length + max(config.lags_sequence),
        future_length=config.prediction_length,
        time_series_fields=['time_features', 'observed_mask'],
    )

def _pred_fields(config):
    f = ['past_time_features', 'past_values', 'past_observed_mask', 'future_time_features']
    if config.num_static_categorical_features > 0:
        f.append('static_categorical_features')
    return f

def create_backtest_dataloader(config, freq, data, batch_size):
    tr = create_transformation(freq, config)
    sp = create_instance_splitter(config, 'validation')
    return as_stacked_batches(sp.apply(tr.apply(data), is_train=True),
                              batch_size=batch_size, output_type=torch.tensor,
                              field_names=_pred_fields(config))

print('Pipeline GluonTS prête.')

## 7 — Reconstruction modèle + chargement checkpoint RunC2

In [ ]:
from src.models.softcam_transformer_v4 import (
    SoftCAMTransformerV4Config,
    SoftCAMTransformerV4ForPrediction,
)

cfg = SoftCAMTransformerV4Config(
    prediction_length=PREDICTION_LENGTH,
    context_length=CONTEXT_LENGTH,
    lags_sequence=lags_sequence,
    num_time_features=len(time_features) + 1,
    num_static_categorical_features=1,
    cardinality=[len(train_dataset)],
    embedding_dimension=EMBEDDING_DIM,
    encoder_layers=ENCODER_LAYERS,
    decoder_layers=DECODER_LAYERS,
    d_model=D_MODEL,
    encoder_attention_heads=N_HEADS,
    decoder_attention_heads=N_HEADS,
    encoder_ffn_dim=max(D_MODEL, 32),
    decoder_ffn_dim=max(D_MODEL, 32),
    dropout=DROPOUT,
    use_evidence_layer=USE_EVIDENCE_LAYER,
    evidence_mix=0.05,
    alpha_l1=ALPHA_L1,
    beta_l2=BETA_L2,
    gamma_entropy=GAMMA_ENTROPY_TARGET,
)

model = SoftCAMTransformerV4ForPrediction(cfg).to(device)

state = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(state['model'])
model.eval()

print(f'Checkpoint RunC2 chargé.')
print(f'  num_epochs train         = {state.get("num_epochs", "?")}')
print(f'  seed                     = {state.get("seed", "?")}')
print(f'  gate_strength_final saved = {state.get("gate_strength_final", "?")}')
print(f'  Paramètres modèle        : {sum(p.numel() for p in model.parameters()):,}')

## 8 — Fonction d'évaluation

In [ ]:
from sklearn.metrics import r2_score
from scipy.stats import spearmanr

@torch.no_grad()
def evaluate_at_strength(model, gate_strength, dataloader, dataset, config, device):
    model.ev_layer_v4.gate_strength = float(gate_strength)
    model.eval()
    forecasts = []
    for b in dataloader:
        out = model.generate(
            static_categorical_features=b['static_categorical_features'].to(device)
                if config.num_static_categorical_features > 0 else None,
            past_time_features=b['past_time_features'].to(device),
            past_values=b['past_values'].to(device),
            future_time_features=b['future_time_features'].to(device),
            past_observed_mask=b['past_observed_mask'].to(device),
        )
        forecasts.append(out.sequences.cpu().numpy())
    forecasts = np.vstack(forecasts)
    forecast_median = np.median(forecasts, axis=1)

    r2s, spears = [], []
    for ts_idx in range(len(dataset)):
        target = np.array(dataset[ts_idx]['target'])
        actual = target[-config.prediction_length:]
        pred   = forecast_median[ts_idx]
        r2s.append(r2_score(actual, pred))
        rho, _ = spearmanr(actual, pred)
        spears.append(rho)
    return float(np.mean(r2s)), float(np.mean(spears)), r2s, spears

test_loader = create_backtest_dataloader(cfg, FREQ, test_dataset, BATCH_SIZE_TEST)
print('Fonction d\'évaluation prête.')

## 9 — Sweep gate_strength

In [ ]:
sweep_results = []

for s in GATE_STRENGTH_VALUES:
    print(f'\n── gate_strength = {s:.2f} ──')
    r2, spear, r2s, spears = evaluate_at_strength(model, s, test_loader, test_dataset, cfg, device)
    sweep_results.append({
        'gate_strength': s,
        'r2_mean':       r2,
        'spearman_mean': spear,
        'r2_per_series': r2s,
        'spearman_per_series': spears,
    })
    flag = '✅' if r2 > 0 else ('⚠️' if r2 > -1 else '❌')
    print(f'  R²       = {r2:+.4f}   {flag}')
    print(f'  Spearman = {spear:+.4f}')
    print(f'  Per-series R² : {[f"{x:+.3f}" for x in r2s]}')

print('\n' + '=' * 60)
print('Sweep terminé.')
print('=' * 60)

## 10 — Tableau + courbes R²(s), Spearman(s)

In [ ]:
df_sweep = pd.DataFrame([
    {'gate_strength': r['gate_strength'],
     'R²': r['r2_mean'],
     'Spearman': r['spearman_mean'],
     'verdict': '✅ PASS' if (r['r2_mean'] >= GATE_R2 and r['spearman_mean'] >= GATE_SPEAR)
                else ('⚠️ R²>0' if r['r2_mean'] > 0 else '❌ FAIL')}
    for r in sweep_results
])
print(df_sweep.to_string(index=False))

ss  = [r['gate_strength']  for r in sweep_results]
r2v = [r['r2_mean']        for r in sweep_results]
spv = [r['spearman_mean']  for r in sweep_results]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))

ax1.plot(ss, r2v, 'o-', color='steelblue', lw=2, markersize=8, label='R² (sweep)')
ax1.axhline(B5_R2,    color='green', ls='--', alpha=0.6, label=f'B5 ref ({B5_R2:.3f})')
ax1.axhline(FAYAM_R2, color='gray',  ls='--', alpha=0.4, label=f'FAYAM ({FAYAM_R2:.3f})')
ax1.axhline(0,        color='black', ls=':',  alpha=0.4)
ax1.axhline(GATE_R2,  color='red',   ls=':',  alpha=0.5, label=f'GATE H1.C ({GATE_R2})')
ax1.set_xlabel('gate_strength s'); ax1.set_ylabel('R²')
ax1.set_title(f'{RUN_NAME} — R² vs gate_strength')
ax1.legend(fontsize=8, loc='lower left'); ax1.grid(alpha=0.3)

ax2.plot(ss, spv, 'o-', color='firebrick', lw=2, markersize=8, label='Spearman')
ax2.axhline(B5_SPEAR,    color='green', ls='--', alpha=0.6, label=f'B5 ref ({B5_SPEAR:.3f})')
ax2.axhline(FAYAM_SPEAR, color='gray',  ls='--', alpha=0.4, label=f'FAYAM ({FAYAM_SPEAR:.3f})')
ax2.axhline(0,           color='black', ls=':',  alpha=0.4)
ax2.axhline(GATE_SPEAR,  color='red',   ls=':',  alpha=0.5, label=f'GATE H1.C ({GATE_SPEAR})')
ax2.set_xlabel('gate_strength s'); ax2.set_ylabel('Spearman')
ax2.set_title(f'{RUN_NAME} — Spearman vs gate_strength')
ax2.legend(fontsize=8, loc='lower left'); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{DRIVE_BASE}/figures/sweep_curve.png', dpi=150)
plt.show()

## 11 — Verdict + sauvegarde JSON

In [ ]:
best = max(sweep_results, key=lambda r: r['r2_mean'])
worst = min(sweep_results, key=lambda r: r['r2_mean'])

results = {
    'run':       RUN_NAME,
    'checkpoint': CKPT_PATH,
    'description': 'Sweep gate_strength a l\'inference sur checkpoint RunC2 v4.',
    'gate_strength_values': GATE_STRENGTH_VALUES,
    'sweep': sweep_results,
    'best_strength':  best['gate_strength'],
    'best_r2':        best['r2_mean'],
    'best_spearman':  best['spearman_mean'],
    'worst_strength': worst['gate_strength'],
    'worst_r2':       worst['r2_mean'],
    'references': {
        'fayam': {'r2': FAYAM_R2, 'spearman': FAYAM_SPEAR},
        'b5_v3': {'r2': B5_R2,    'spearman': B5_SPEAR},
        'c1_v4': {'r2': C1_R2,    'spearman': C1_SPEAR},
        'c2_v4': {'r2': C2_R2,    'spearman': C2_SPEAR},
    },
}

with open(f'{DRIVE_BASE}/results/sweep_metrics.json', 'w') as f:
    json.dump(results, f, indent=2, default=str)

print('=' * 68)
print(f'  VERDICT — {RUN_NAME}')
print('=' * 68)
print(f'  Meilleur  : s={best["gate_strength"]:.2f}  R²={best["r2_mean"]:+.4f}  ρ={best["spearman_mean"]:+.4f}')
print(f'  Pire      : s={worst["gate_strength"]:.2f}  R²={worst["r2_mean"]:+.4f}  ρ={worst["spearman_mean"]:+.4f}')
print()

r2_at_0 = next(r['r2_mean'] for r in sweep_results if r['gate_strength'] == 0.0)
r2_at_1 = next(r['r2_mean'] for r in sweep_results if r['gate_strength'] == 1.0)

print('  Interprétation :')
if r2_at_0 > 0.30:
    print(f'  ✅ À s=0, R²={r2_at_0:+.4f} > seuil GATE → le Transformer interne est sain.')
    print(f'     La cause du fail C1/C2 est BIEN le gate en autorégression.')
elif r2_at_0 > 0:
    print(f'  🟡 À s=0, R²={r2_at_0:+.4f} > 0 mais < seuil → Transformer partiellement sain.')
    print(f'     Le warm-up de C2 a peut-être pollué le Transformer.')
else:
    print(f'  ❌ À s=0, R²={r2_at_0:+.4f} < 0 → le Transformer lui-même est cassé.')
    print(f'     Le problème est plus profond que le gate.')

print()
if best['r2_mean'] >= GATE_R2:
    print(f'  ✅ Sweet spot trouvé à s={best["gate_strength"]:.2f} → utiliser cette valeur en inférence.')
elif best['r2_mean'] > 0:
    print(f'  🟡 Sweet spot positif mais < seuil GATE → pas suffisant.')
else:
    print(f'  ❌ Aucun s ne donne R²>0 → architecture v4 multiplicative inviable telle quelle.')

print('=' * 68)